## 1. Installation des dépendances

Les commandes suivantes installent les bibliothèques nécessaires pour l'extraction de texte, la génération de réponses, et la recherche documentaire :

- **farm-haystack** : Permet de créer des systèmes de recherche documentaire basés sur l'intelligence artificielle. Cette bibliothèque offre des outils pour l'indexation, la recherche et la récupération de documents à partir de données non structurées.
  
- **sentence-transformers** : Utilisée pour créer des **embeddings de phrases** qui permettent de réaliser des recherches sémantiques. Les embeddings transforment le texte en vecteurs numériques, facilitant ainsi la comparaison de similarité entre différentes phrases ou documents.

- **transformers** : Permet d'utiliser des modèles pré-entraînés pour des tâches de traitement du langage naturel (NLP), comme la génération de texte, la traduction, et la réponse à des questions. Les modèles pré-entraînés de **Hugging Face** sont populaires pour leur performance sur de nombreuses tâches NLP.

- **PyPDF2** : Bibliothèque utilisée pour **extraire du texte** des fichiers PDF. Elle permet de lire des fichiers PDF et d'en extraire le contenu textuel page par page, ce qui est essentiel pour le traitement de documents non structurés.


In [1]:
!pip install --upgrade farm-haystack
!pip install --upgrade sentence-transformers
!pip install --upgrade transformers

In [3]:
!pip install PyPDF2 sentence-transformers faiss-cpu farm-haystack transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.5/27.5 MB 53.4 MB/s eta 0:00:00


## 2. Import des bibliothèques

Les modules nécessaires sont importés pour charger les modèles de génération de texte, effectuer des recherches dans les documents, et manipuler les fichiers PDF.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from haystack.nodes import EmbeddingRetriever
from haystack.document_stores import InMemoryDocumentStore
import PyPDF2
import re

## 3. Fonction `extract_text_from_pdf`

Cette fonction permet d'extraire le texte d'un fichier PDF en utilisant **PyPDF2**. Elle ouvre le fichier PDF, lit chaque page et extrait son contenu textuel.

In [ ]:
# Fonction pour extraire le texte d'un PDF
def extract_text_from_pdf(pdf_path):
    """Extrait le texte d'un fichier PDF."""
    text = ""
    with open(pdf_path, "rb") as pdf_file:
        reader = PyPDF2.PdfReader(pdf_file)
        for page in reader.pages:
            text += page.extract_text()
    return text

## 4. Fonction `index_documents`

Cette fonction indexe les documents dans un **InMemoryDocumentStore**, un stockage en mémoire pour les documents. Elle utilise un modèle d'**embedding** pour transformer chaque document en vecteurs. Ces vecteurs permettent une recherche rapide et efficace dans le système de gestion de documents.

In [87]:
# Étape 1 : Indexation des documents
def index_documents(documents):
    """Indexe les documents en utilisant un Document Store en mémoire."""
    document_store = InMemoryDocumentStore(embedding_dim=384)

    # Embedding model pour créer les vecteurs
    retriever = EmbeddingRetriever(
        document_store=document_store,
        embedding_model="sentence-transformers/all-MiniLM-L6-v2"
        #embedding_model="sentence-transformers/paraphrase-MiniLM-L6-v2"
    )

    # Conversion des documents en format Haystack
    docs_to_index = [{"content": doc} for doc in documents]

    # Écriture des documents dans le document store
    document_store.write_documents(docs_to_index)

    # Mise à jour des embeddings pour les documents indexés
    document_store.update_embeddings(retriever)

    print(f"Nombre de documents indexés : {document_store.get_document_count()}")
    print(f"Nombre d'embeddings : {document_store.get_embedding_count()}")

    return document_store, retriever

## 5. Fonction `retrieve_documents`

Cette fonction recherche les documents les plus pertinents en fonction de la requête de l'utilisateur. Elle utilise le retriever pour effectuer une recherche dans le **Document Store** et renvoie les **top_k** documents les plus proches du texte de la requête. Cette approche est basée sur la recherche sémantique, qui permet de trouver les documents les plus pertinents en fonction du contenu, et non simplement de mots-clés.

In [ ]:
# Étape 2 : Recherche dans les documents
def retrieve_documents(query, retriever, document_store, top_k=3):
    """Recherche les k documents les plus pertinents."""
    retrieved_docs = retriever.retrieve(query=query, top_k=top_k)
    return [doc.content for doc in retrieved_docs]

## 6. Fonction `generate_response`

Cette fonction génère une réponse détaillée à partir des documents récupérés. Elle utilise un modèle de génération pré-entraîné de Hugging Face, comme `google/flan-t5-large`, pour créer une réponse basée sur les informations contextuelles extraites des documents récupérés.

In [89]:
# Étape 3 : Génération avec Hugging Face
def generate_response(query, retrieved_docs):
    """Génère une réponse à partir des documents récupérés."""

    model_name = "google/flan-t5-large"
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    context = " ".join(retrieved_docs)
    #prompt = f"Generate your own detailed response to the question:\nQuestion: {query} \nResponse:"
    prompt = f"Answer the following question using the given context, in a detailed manner:\n{context}\n\nQuestion: {query}\nResponse:"
    #prompt = f"Given the following information, please provide a detailed answer, adding any relevant information you know:\n{context}\n\nQuestion: {query}\nResponse:"

    # Générer une réponse
    inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)
    outputs = model.generate(**inputs, max_length=300)
    #print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!",retrieved_docs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## 7. Exécution principale

Cette section charge un fichier PDF, extrait son texte, le divise en morceaux, puis indexe les documents. Elle exécute ensuite des requêtes pour récupérer des documents pertinents et générer des réponses à partir du modèle de génération.

In [90]:
if __name__ == "__main__":
    #pdf_path = "/content/PdfDataTest.pdf"
    pdf_path = "/content/monopoly.pdf"

    # Extraire le texte du PDF
    pdf_text = extract_text_from_pdf(pdf_path)

    # prise en charge des expressions regulière
    documents = re.split(r'(\d+\.)', pdf_text)  # Diviser à chaque numéro de section (1., 2., etc.)

    documents = [documents[i] + documents[i+1] for i in range(0, len(documents)-1, 2)]

    # Indexation des documents
    document_store, retriever = index_documents(documents)

queries = [
    "What is the name of the game?",
    "Explain the Speed Die rules in Monopoly.",
    "How does a player buy properties in Monopoly?",
    "What happens when you land on 'Go to Jail'?",
    "How do you get out of Jail in Monopoly?",
    "What are the rules for bankrupt players in Monopoly?",
    "How does the auction process work in Monopoly?",
    "Explain how properties are mortgaged in Monopoly.",
    "What happens when you land on 'Income Tax' in Monopoly?",
    "Give me the rules about Chance and Community Chest cards.",
    "What are the differences between the classic and Speed Die rules?",

]

for query in queries:
    retrieved_docs = retrieve_documents(query, retriever, document_store)
    response = generate_response(query, retrieved_docs)
    print(f"Generated response for query: '{query}'")
    print(response)
    print("\n------------------------------\n")

Updating Embedding:   0%|          | 0/7 [00:00<?, ? docs/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Documents Processed: 10000 docs [00:00, 94730.23 docs/s]

Nombre de documents indexés : 7
Nombre d'embeddings : 7


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Generated response for query: 'What happens when you land on 'Income Tax' in Monopoly?'
the value of one die.

------------------------------



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated response for query: 'Give me the rules about Chance and Community Chest cards.'
've never played the classic MONOPOLY game, refer to the Classic Rules beginning on the next page. If you already know how to play and want to use the Speed Die, just read the section below for the additional Speed Die rules. 1. CLASSIC MONOPOW RULES OBJECT: The object of the game IS to become the wealthiest player through buying, renting and selling property. PREPARATION: Place the board on a table and put the Chance and Community Chest cards facedown on their allotted spaces on the board. Each player chooses one token to represent himther while traveling around the board. Each player is given $1,500 divided as follows: P each of $500s, $100 and $50; 6 $40; 5 each of $105, $5 and $Is. All remaining money and other equipment go to the Bank. BANKER. Select as Banker a player who will also make a good Auctioneer A Banker who plays n the game must keep hislher personal funds separate from those of th

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated response for query: 'What are the differences between the classic and Speed Die rules?'
Do not use the Speed Die until you've landed on or passed over GO for the first time. Once you collect that first $200 salary, you'll use the Speed Die for the rest of the game.

------------------------------

